# Full Pipeline AI Agent


In [2]:
!pip install -q google-genai langchain langchain-google-genai requests pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 707.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 23.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [3]:
import os
import json
import requests

from urllib.parse import urlparse
from google.colab import userdata
from google import genai
from google.genai import types

from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List
from typing import Optional

In [4]:


# --- Intent Clarification Agent output schema ---
class IntentOutput(BaseModel):
    event_type: str
    location: str
    time_period: str
    #event_intensity: Optional[str] = None
    keywords: List[str] = Field(min_length=1)


# --- Nested schema: one article inside EventOutput ---
class ArticleItem(BaseModel):
    title: str
    url: str     # TODO: str
    snippet: str # TODO: str


# --- Event Identification Agent output schema ---
class EventOutput(BaseModel):
    event_name: str # TODO: str
    event_type: str # TODO: str
    location:  str # TODO: str
    date: str     # TODO: str
    deaths: int = 0
    injuries: int = 0
    displaced_people: int = 0
    infrastructure_damage: str = "unknown"
    summary: str   # TODO: str
    articles: List[ArticleItem] # TODO: List[ArticleItem]


# --- Nested schema: one image inside the enriched event dict ---
class ImageItem(BaseModel):
    title: str        # TODO: str
    url: str          # TODO: str
    thumbnail_url: str # TODO: str
    description: str  # TODO: str
    license: str     # TODO: str
    author: str       # TODO: str
    local_path: str


# --- Situation brief output schema (Summarization Agent) ---
class SituationBrief(BaseModel):
    headline:  str          # TODO: str
    overview:  str        # TODO: str
    severity:  str          # TODO: str
    affected_population: str # TODO: str
    response:  str          # TODO: str
    key_facts: List[str] = []
    sources_used: List[str] = Field(min_length=1)       # TODO: List[str] = Field(min_length=1)

    @field_validator("headline")
    @classmethod
    def headline_max_20_words(cls, v: str) -> str:

        word_count = len(v.split())

        if word_count > 20:
            raise ValueError("Headline must be 20 words or fewer")

        return v

# --- Timeline Item and output schema ---
class TimelineItem(BaseModel):
    date: str
    time: Optional[str] = None
    event: str

class TimelineOutput(BaseModel):
    timeline: list[TimelineItem]




In [5]:
def validated_intent_agent_output(raw_json_str: str):
    data = json.loads(raw_json_str)

    try:
        validated = IntentOutput.model_validate(data)
        return validated.model_dump()

    except ValidationError as e:
        print("Intent validation failed:")
        print(e)
        return None


def validated_event_agent_output(raw_json_str: str) -> dict:
    raw = raw_json_str.strip()

    if raw.startswith("```"):
        raw = raw.strip("`").strip()

        if raw.startswith("json"):
            raw = raw[4:].strip()

    data = json.loads(raw)

    try:
        validated = EventOutput.model_validate(data)
        return validated.model_dump()

    except ValidationError as e:
        raise ValueError(
            f"Event agent returned invalid data: {e}"
        )


def validated_image_item(raw_image_dict: dict):
    try:
        validated = ImageItem.model_validate(raw_image_dict)
        return validated.model_dump()

    except ValidationError as e:
        print(f"Invalid image skipped: {e}")
        return None


def validate_images_list(images: list) -> list:
    valid_images = []

    for image in images:
        validated = validated_image_item(image)

        if validated is not None:
            valid_images.append(validated)

    return valid_images


def validated_situation_brief(raw_json_str: str) -> dict:
    raw = raw_json_str.strip()

    if raw.startswith("```"):
        raw = raw.strip("`").strip()

        if raw.startswith("json"):
            raw = raw[4:].strip()

    data = json.loads(raw)

    try:
        validated = SituationBrief.model_validate(data)
        return validated.model_dump()

    except ValidationError as e:
        raise ValueError(
            f"Situation brief validation failed: {e}"
        )


def validated_timeline_output(raw_json_str):
    try:
        data = json.loads(raw_json_str)

        validated = TimelineOutput.model_validate(data)

        return validated.model_dump()

    except json.JSONDecodeError as e:
        print("Timeline JSON parsing failed:")
        print(e)
        return None

    except ValidationError as e:
        print("Timeline validation failed:")
        print(e)
        return None

In [6]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print(f"✓ GOOGLE_API_KEY loaded  ({GOOGLE_API_KEY[:6]}...)")
else:
    raise ValueError(
        "GOOGLE_API_KEY not found in Colab Secrets.\n"
        "Go to 🔑 Secrets, add GOOGLE_API_KEY, and enable notebook access."
    )


client = genai.Client(api_key=GOOGLE_API_KEY)


✓ GOOGLE_API_KEY loaded  (AQ.Ab8...)


In [7]:
INTENT_SYSTEM_PROMPT = """
You are a disaster event clarification assistant. Your job is to ask the user
targeted questions to identify exactly which natural disaster event they are
referring to. You need three pieces of information:
  1. Event type (e.g. earthquake, flood, wildfire, hurricane)
  2. Location (country, region, or city)
  3. Time period (year, or approximate date range)
  4. Event intensity, only if the user mentions it or it helps identify the event (for example earthquake magnitude or hurricane category)

Ask only ONE question per response.

If the user does not mention event intensity and it is not important to the event, do not ask asbout it.

Once you have all three pieces of
information, respond ONLY with a JSON object in this exact format and nothing else:
{
  "event_type": "...",
  "location": "...",
  "time_period": "...",
  "keywords": ["...", "...", "..."]
}
"""

print("✓ System prompt defined.")



✓ System prompt defined.


In [8]:
def run_intent_agent():
    # TODO: create a Gemini chat session using client.chats.create()
    #       pass model="gemini-2.0-flash" and system_instruction=SYSTEM_PROMPT
    chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=INTENT_SYSTEM_PROMPT
    )
)

    print("Describe the disaster event you want to know about (type 'quit' to exit):\n")

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue
        if user_input.lower() == "quit":
            print("Exiting.")
            break

        # TODO: send user_input to the chat session and get the reply text
        reply = chat.send_message(user_input).text

        # Strip markdown code fences if the model wraps its JSON
        if reply.startswith("```"):
            reply = reply.strip("`").strip()
            if reply.startswith("json"):
                reply = reply[4:].strip()

        # If the model returned a JSON object, we're done
        if reply.startswith("{"):

          event_query = validated_intent_agent_output(reply)

          if event_query is not None:

              print("\n✓ Event identified and validated:")
              print(json.dumps(event_query, indent=2))

              return event_query

          else:
              print("Agent output was invalid. Please provide the event information again.")

        else:

            print(f"Agent: {reply}\n")

print("✓ run_intent_agent() defined — run the next cell to start a conversation.")

✓ run_intent_agent() defined — run the next cell to start a conversation.


In [9]:
EVENT_SYSTEM_PROMPT = """
You are a disaster event research agent. Given a structured query, use Google
Search to find accurate, current information about the specified natural disaster.

Return ONLY a valid JSON object — no markdown, no extra text — with these fields:
{
  "event_name": "commonly used name for the event",
  "event_type": "type of disaster",
  "location": "specific locations affected",
  "date": "date or date range of the event",
  "deaths": "reported number of deaths, or 0 if unknown",
  "injuries": "reported number of injuries, or 0 if unknown",
  "displaced_poeple": "reported number of displaced people, or 0 if unknown",
  "infrastructure_damage": "low, moderate, major, or severe",
  "summary": "2-3 sentence factual overview of the event",
  "articles": [
    {"title": "article headline", "url": "article URL", "snippet": "1 sentence description"},
    ...
  ]
}
If information for deaths, injuries, or displaced_people is unavailable,
use 0. Do not omit any fields.

For infrastructure_damage, use one of:
"low", "moderate", "major", "severe", or "unknown".

Include up to 5 articles. 5 articles maximum. Populate articles from the sources you searched.
"""

print("✓ System prompt defined.")

✓ System prompt defined.


In [10]:
def run_event_agent(event_query: dict) -> dict:
    # TODO: Build a prompt string that passes event_type, location,
    #       time_period, and keywords from event_query to the model.
    prompt = f"""
Identify an event using the following search criteria:

Event type: {event_query.get("event_type", "")}
Location: {event_query.get("location", "")}
Time period: {event_query.get("time_period", "")}
Keywords: {event_query.get("keywords", [])}

Return the result as valid JSON following the format described in the
system instructions. Do not include explanations outside the JSON.
"""


    # TODO: Call client.models.generate_content() with:
    #       model="gemini-2.0-flash"
    #       contents=prompt
    #       config=types.GenerateContentConfig(
    #           system_instruction=SYSTEM_PROMPT,
    #           tools=[types.Tool(google_search=types.GoogleSearch())]
    #       )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=EVENT_SYSTEM_PROMPT,
            tools=[
                types.Tool(
                    google_search=types.GoogleSearch()
                )
            ]
        )
    )

    # Extract grounding chunks (real article URLs Gemini searched)
    sources = []
    grounding = response.candidates[0].grounding_metadata
    if grounding and grounding.grounding_chunks:
        for chunk in grounding.grounding_chunks:
            if chunk.web:
                sources.append({"title": chunk.web.title, "url": chunk.web.uri, "snippet": ""})

    # TODO: Parse the model's text response as JSON.
    #       Strip markdown code fences first, then call json.loads().
    raw = response.text.strip()

    if raw.startswith("```json"):
        raw = raw[len("```json"):]

    if raw.startswith("```"):
        raw = raw[len("```"):]

    if raw.endswith("```"):
        raw = raw[:-len("```")]

    raw = raw.strip()
    event_data = validated_event_agent_output(raw)

    # Merge grounding sources into articles list
    if not event_data.get("articles") and sources:
      event_data["articles"] = sources

    elif sources:

        existing_urls = {
            article["url"]
            for article in event_data["articles"]
        }

        for source in sources:

            if source["url"] not in existing_urls:
                event_data["articles"].append(source)

    event_data["articles"] = event_data["articles"][:5]

    return event_data

print("✓ run_event_agent() defined — run the next cell to call the agent.")

✓ run_event_agent() defined — run the next cell to call the agent.


# Disaster Severity Score

This enhancement calculates a consistent disaster severity score from 1-5 using the impact data collected by the Event Research Agent.

In [11]:
def calculate_severity(event_data):
  score = 1 #score is from 1-5

  deaths = event_data["deaths"]
  injuries = event_data["injuries"]
  displaced = event_data["displaced_people"]
  damage = event_data["infrastructure_damage"].lower()

  # deaths
  if deaths > 1000:
    score += 2
  elif deaths >= 150:
    score += 1

  # injuries
  if injuries >= 1000:
    score += 1

  # displaced people
  if displaced >= 10000:
    score += 1

  # infrastructure damage
  if damage == "severe":
        score += 1
  elif damage == "major":
        score += 0.5

  return min(round(score), 5)

In [12]:
def get_severity_level(score):
  if score == 1:
    return "Low"
  elif score == 2:
    return "Moderate"
  elif score == 3:
    return "Significant"
  elif score == 4:
    return "High"
  else:
    return "Extreme"

In [13]:
def add_severity_score(event_data):
  severity_score = calculate_severity(event_data)
  severity_level = get_severity_level(severity_score)

  event_data["severity_score"] = severity_score
  event_data["severity_level"] = severity_level

  return event_data

In [14]:
!pip install requests --quiet

import requests
import json

WIKIMEDIA_API_URL = "https://commons.wikimedia.org/w/api.php"

# Wikimedia requires a descriptive User-Agent on every request, or it returns HTTP 403.
# See: https://meta.wikimedia.org/wiki/User-Agent_policyprint("✓ requests imported. No API key needed for Wikimedia.")

HEADERS = {"User-Agent": "EventInsightBot/1.0 (https://example.com/contact; student-project)"}

In [15]:
def search_event_images(event_name: str, location: str, max_images: int = 5) -> list:
    """Search Wikimedia Commons and return a list of image dicts."""

    MAX_IMAGES = 5

    search_query = event_name

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": search_query,
        "srnamespace": 6,
        "srlimit": MAX_IMAGES,
        "format": "json",
    }

    search_resp = requests.get(
        WIKIMEDIA_API_URL,
        params=search_params,
        headers=HEADERS,
    )

    search_data = search_resp.json()

    print("Search query:", search_query)
    print("Search results:", search_data)

    file_titles = [
        item["title"]
        for item in search_data["query"]["search"]
    ]

    if not file_titles:
        return []

    info_params = {
        "action": "query",
        "titles": "|".join(file_titles),
        "prop": "imageinfo",
        "iiprop": "url|extmetadata",
        "format": "json",
    }

    info_resp = requests.get(
        WIKIMEDIA_API_URL,
        params=info_params,
        headers=HEADERS,
    )

    info_data = info_resp.json()

    images = []
    for page in info_data.get("query", {}).get("pages", {}).values():
        if "imageinfo" not in page:
            continue
        info = page["imageinfo"][0]
        ext  = info.get("extmetadata", {})
        images.append({
            "title":         page["title"],
            "url":           info.get("url", ""),
            "thumbnail_url": info.get("thumburl", info.get("url", "")),
            "description":   ext.get("ImageDescription",  {}).get("value", ""),
            "license":       ext.get("LicenseShortName",  {}).get("value", ""),
            "author":        ext.get("Artist",             {}).get("value", ""),
        })

    return images

In [16]:
def download_image(image_url: str, output_dir: str = "downloaded_images") -> str:
    """Download an image from a URL and return the saved file path."""

    os.makedirs(output_dir, exist_ok=True)

    clean_url = image_url.split("?")[0]
    filename = os.path.basename(urlparse(clean_url).path)

    file_path = os.path.join(output_dir, filename)

    response = requests.get(image_url, headers=HEADERS)
    response.raise_for_status()

    with open(file_path, "wb") as f: f.write(response.content)

    return file_path


In [17]:
'''
def collect_event_images(event_data: dict, max_images: int = 5) -> dict:
    """Search for event images, download them, and add them to event_data."""

    event_name = event_data.get("event_name")
    location = event_data.get("location", "")

    if not event_name:
        print("No 'event_name' found in event_data.")
        print("Available keys:", event_data.keys())
        print("event_data:", json.dumps(event_data, indent=2))
        return event_data

    images = search_event_images(
        event_data["event_name"],
        event_data["location"],
        max_images
    )

    for image in images:
        image_url = image["url"]

        try:
            local_path = download_image(image_url)
            image["local_path"] = local_path

        except Exception as error:
            print(f"Could not download {image_url}: {error}")
            image["local_path"] = None

    images = validate_images_list(images)

    enriched_event = event_data.copy()
    enriched_event["images"] = images

    return enriched_event

'''

def collect_event_images(
    event_data: dict,
    max_images: int = 5
) -> dict:

    images = search_event_images(
        event_data["event_name"],
        event_data["location"],
        max_images
    )

    for image in images:

        image_url = image["url"]

        try:
            local_path = download_image(image_url)
            image["local_path"] = local_path

        except Exception as error:
            print(f"Could not download {image_url}: {error}")
            image["local_path"] = None

    # Workshop 6 validation
    images = validate_images_list(images)

    enriched_event = event_data.copy()
    enriched_event["images"] = images

    return enriched_event

In [18]:
SUMMARY_SYSTEM_PROMPT = """
You are a disaster event summarisation agent. You will be given a set of article
snippets about a natural disaster event. Your job is to synthesise these into a
structured situation brief.

Return ONLY a valid JSON object — no markdown, no extra text — with these fields:
{
  "headline": "A single-sentence headline for the event (max 20 words)",
  "overview": "2-3 sentence factual overview of what happened, where, and when",
  "severity": "2-3 sentences explaining the disaster impact, including casualties, damage scale, event intensity, and the calculated severity score if provided",
  "affected_population": "1-2 sentences on the number of people directly affected",
  "response": "1-2 sentences on emergency response, international aid, or government action",
  "key_facts": "A list of 3-5 bullet-point facts about the event",
  "sources_used": ["url1", "url2", ...]
}
If the snippets lack detail for a field, draw on your knowledge of the event to
complete it. Always populate sources_used with the URLs provided.
"""

print("✓ System prompt defined.")

✓ System prompt defined.


In [19]:
def build_article_context(articles: list) -> str:
    """Format the articles list into a readable context string for the model."""
    if not articles:
        return "No article snippets available."

    lines = []
    for i, article in enumerate(articles, 1):
        lines.append(f"Article {i}: {article.get('title', 'Untitled')}")
        lines.append(f"URL: {article.get('url', '')}")
        lines.append(f"Snippet: {article.get('snippet', '')}")
        lines.append("")

    return "\n".join(lines)




In [20]:
def run_summarization_agent(event_data: dict) -> dict:
    # TODO: Call build_article_context() with event_data["articles"]
    article_context = build_article_context(event_data["articles"])

    # TODO: Build a prompt string that passes event_name, location, date,
    #       and the article_context to the model.
    prompt = f"""
      Event Name: {event_data['event_name']}
      Location: {event_data['location']}
      Date: {event_data['date']}
      Severity Score: {event_data['severity_score']}/5
      Severity Level: {event_data['severity_level']}
      Articles: {article_context}

      Using the event information and article snippets above,
      create a structured situation brief.
      """


    # TODO: Call client.models.generate_content() with:
    #       model="gemini-2.0-flash"
    #       contents=prompt
    #       config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(system_instruction=SUMMARY_SYSTEM_PROMPT)
      )

    # Strip markdown code fences if the model wraps its JSON, then parse
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").strip()
        if raw.startswith("json"):
            raw = raw[4:].strip()

    situation_brief = validated_situation_brief(raw)
    event_data["situation_brief"] = situation_brief
    return event_data


print("✓ run_summarization_agent() defined — run the next cell to generate the brief.")

✓ run_summarization_agent() defined — run the next cell to generate the brief.


# Timeline Generator Agent

In [21]:
TIMELINE_SYSTEM_PROMPT = """
You are a disaster event timeline agent.
You will be given information about a natural disaster event, including research results and article snippets.

Your job is to identify the most important events related to the disaster and arrange them in chronological order.

Return ONLY a valid JSON object — no markdown, no extra text - with the following fields:
{
  "timeline": [
    {
      "date": "date or approximate date",
      "time": "time of the event including timezone, or null if unavailable",
      "event": "short description of what happened"
    }
  ]

}

Include 3-6 important events when enough information is available. If possible more than 3.

Events may include:
- the disaster beginning or occuring
- major impacts
- evacuations
- emergency declarations
- rescue or government responses
- important secondary disasters
- major recovery developments

For events where an exact or approximate time is available, include the time
and timezone.

If the source information does not provide a time, use null.
Do NOT guess or invent a time.

Arrange the events chronologically by date and time.

Keep each event description 1-2 sentences.

Only include events supported by the provided disaster information.
Do not invent dates or events.

"""


In [22]:
def run_timeline_agent(event_data: dict) -> dict:

  article_context = build_article_context(event_data["articles"])

  prompt = f"""
  Event Name: {event_data['event_name']}
  Event Type: {event_data['event_type']}
  Location: {event_data['location']}
  Date: {event_data['date']}
  Summary: {event_data['summary']}

  Articles: {article_context}

  Using the event information and article snippets above,
  create a structured chronological timeline for this disaster event.
  """

  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt,
      config=types.GenerateContentConfig(system_instruction=TIMELINE_SYSTEM_PROMPT)
    )

  raw = response.text.strip()

  if raw.startswith("```"):
    raw = raw.strip("`").strip()
    if raw.startswith("json"):
      raw = raw[4:].strip()

  timeline_data = validated_timeline_output(raw)

  if timeline_data is None:
    print("Timeline could not be validated.")
    return event_data

  event_data["timeline"] = timeline_data["timeline"]

  return event_data

print("✓ run_timeline_agent() defined.")

# Implement google search grounding


✓ run_timeline_agent() defined.


In [23]:
def format_number(value):
    if isinstance(value, (int, float)):
        return f"{value:,}"
    return str(value) if value is not None else "Unknown"


def display_disaster_report(data):
    brief = data.get("situation_brief", {})

    # -----------------------------
    # REPORT HEADER
    # -----------------------------
    print("\n")
    print("=" * 70)
    print("                     DISASTER SITUATION REPORT")
    print("=" * 70)

    print(f"\n{data.get('event_name', 'Unknown Event').upper()}")
    print(
        f"{data.get('event_type', 'Unknown')} | "
        f"{data.get('date', 'Unknown Date')}"
    )

    print(
        f"\nSEVERITY: {data.get('severity_level', 'Unknown').upper()} "
        f"({data.get('severity_score', '?')}/5)"
    )

    # -----------------------------
    # OVERVIEW
    # -----------------------------
    print("\n" + "-" * 70)
    print("OVERVIEW")
    print("-" * 70)

    overview = brief.get(
        "overview",
        data.get("summary", "No overview available.")
    )

    print(f"\n{overview}")

    # -----------------------------
    # AFFECTED AREA
    # -----------------------------
    print("\n" + "-" * 70)
    print("AFFECTED AREA")
    print("-" * 70)

    print(f"\n{data.get('location', 'Unknown')}")

    # -----------------------------
    # IMPACT SUMMARY
    # -----------------------------
    print("\n" + "-" * 70)
    print("IMPACT SUMMARY")
    print("-" * 70)

    print(f"\nDeaths:                 {format_number(data.get('deaths'))}")
    print(f"Injuries:               {format_number(data.get('injuries'))}")
    print(f"Displaced People:       {format_number(data.get('displaced_people'))}")
    print(
        f"Infrastructure Damage:  "
        f"{str(data.get('infrastructure_damage', 'Unknown')).title()}"
    )
    print(f"Severity Score:         {data.get('severity_score', '?')} / 5")

    # -----------------------------
    # KEY FACTS
    # -----------------------------
    print("\n" + "-" * 70)
    print("KEY FACTS")
    print("-" * 70)

    key_facts = brief.get("key_facts", [])

    if key_facts:
        print()
        for fact in key_facts:
            print(f"• {fact}")
    else:
        print("\nNo key facts available.")

    # -----------------------------
    # EVENT TIMELINE
    # -----------------------------
    print("\n" + "-" * 70)
    print("EVENT TIMELINE")
    print("-" * 70)

    timeline = data.get("timeline", [])

    if timeline:
        for item in timeline:
            print(f"\n{item.get('date', 'Unknown Date')}")

            if item.get("time"):
                print(f"Time: {item.get('time')}")

            print(f"→ {item.get('event', 'No event description available.')}")
    else:
        print("\nNo timeline available.")

    # -----------------------------
    # RESPONSE
    # -----------------------------
    print("\n" + "-" * 70)
    print("RESPONSE")
    print("-" * 70)

    print(
        f"\n{brief.get('response', 'No response information available.')}"
    )

    # -----------------------------
    # SOURCES
    # -----------------------------
    print("\n" + "-" * 70)
    print("SOURCES")
    print("-" * 70)

    articles = data.get("articles", [])

    if articles:
        print()
        for i, article in enumerate(articles, start=1):
            print(f"{i}. {article.get('title', 'Unknown Source')}")
    else:
        print("\nNo sources available.")

    # -----------------------------
    # END
    # -----------------------------
    print("\n" + "=" * 70)
    print("                      END OF SITUATION REPORT")
    print("=" * 70)

In [26]:
def run_disaster_agent():
    """
    Run all four stages of the Disaster AI Agent.
    User
    Intent Agent
    Event Research Agent
    Wikimedia Image Collector
    Summarization Agent
    Final Event Data
    """

    print("\n================================")
    print("STEP 1: EVENT CLARIFICATION")
    print("================================\n")

    event_query = run_intent_agent()

    if not event_query:
        return None

    print("\n================================")
    print("STEP 2: EVENT RESEARCH")
    print("================================\n")

    event_data = run_event_agent(event_query)

    print("\n✓ Event research complete:")

    print("\n================================")
    print("STEP 3: SEVERITY ANALYSIS")
    print("================================\n")

    event_data = add_severity_score(event_data)

    print("✓ Severity analysis complete:")
    print("Severity Score:", event_data["severity_score"])
    print("Severity Level:", event_data["severity_level"])

    print("\n================================")
    print("STEP 4: IMAGE COLLECTION")
    print("================================\n")

    event_data = collect_event_images(event_data)

    print(
        f"✓ Collected "
        f"{len(event_data.get('images', []))} image(s)"
    )

    print("\n================================")
    print("STEP 5: SITUATION BRIEF")
    print("================================\n")

    event_data = run_summarization_agent(event_data)

    print("\n================================")
    print("STEP 6: EVENT TIMELINE")
    print("================================\n")

    event_data = run_timeline_agent(event_data)

    print("✓ Event timeline created:")

    for item in event_data["timeline"]:
      if item["time"]:
        print(f"{item['date']} | {item['time']} - {item['event']}")
      else:
        print(f"{item['date']} - {item['event']}")

    print("\n================================")
    print("FINAL DISASTER EVENT DATA")
    print("================================\n")

    display_disaster_report(event_data)


    return event_data

In [27]:
# Run the AI agnet

final_event_data = run_disaster_agent()


STEP 1: EVENT CLARIFICATION

Describe the disaster event you want to know about (type 'quit' to exit):

You: japan 2011 earthquake

✓ Event identified and validated:
{
  "event_type": "earthquake",
  "location": "Japan",
  "time_period": "2011",
  "keywords": [
    "Tohoku",
    "tsunami",
    "Great East Japan Earthquake"
  ]
}

STEP 2: EVENT RESEARCH


✓ Event research complete:

STEP 3: SEVERITY ANALYSIS

✓ Severity analysis complete:
Severity Score: 5
Severity Level: Extreme

STEP 4: IMAGE COLLECTION

Search query: 2011 Tōhoku earthquake and tsunami
Search results: {'batchcomplete': '', 'continue': {'sroffset': 5, 'continue': '-||'}, 'query': {'searchinfo': {'totalhits': 172831}, 'search': [{'ns': 6, 'title': 'File:2011 Tōhoku earthquake and tsunami damage Matsushima, Miyagi.JPG', 'pageid': 33491717, 'size': 597, 'wordcount': 156, 'snippet': 'Creative Commons Attribution-Share Alike <span class="searchmatch">3</span>.0 Unported license. You are free: to share – to copy, distribute